In [1]:
!pip install sentence-transformers==2.2.2
!pip install wordfreq
import re  # For regex-based text processing
import pandas as pd  # For working with dataframes
import numpy as np  # For numerical operations, e.g., z-score calculation
# For tensor operations, using PyTorch
import torch
from tqdm import tqdm  # For progress bars
from scipy.stats import zscore  # For calculating z-scores
from sentence_transformers import SentenceTransformer, util  # For loading the model and calculating similarity
from nltk.corpus import stopwords  # For loading stopwords (ensure 'nltk' data is downloaded)
import argparse  # For command-line argument parsing
import pickle  # For saving and loading embeddings as .pkl files
from nltk.corpus import stopwords
from wordfreq import top_n_list

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125924 sha256=8aeb3c2032411f730cf9faa8c1b00afcc07059d900e135b87feb8db6c171dbeb
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.0 MB/s eta 0:00:00


In [2]:
top100 = top_n_list('de', 100)
def count_top100(s):
    return len([1 for w in s.split() if w in set(top100)])


def config(parser):  
    parser.add_argument('--model_name_or_path', default='/kaggle/input/word2vec-new/word2vec_new.model')
    parser.add_argument('--input_file', default='/kaggle/input/parliament2/speeches_clean.csv')
    parser.add_argument('--output_file', default='speeches_new_emi.csv')
    parser.add_argument('--evidence_lexicon', default='/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv')
    parser.add_argument('--intuition_lexicon', default='/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv')
    parser.add_argument('--save_embeddings', action="store_true")
    parser.add_argument('--smoke_test', action="store_true")
    parser.add_argument('--text_column', type=str, default='speechContent')
    parser.add_argument('--compression_type', type=str, default='infer')
    parser.add_argument('--length_threshold', type=int, default=10)
    parser.add_argument('--tab_delimiter', action="store_true")
    parser.add_argument('--chunk_text', action="store_true")
    parser.add_argument('--min_chunk_length', type=int, default=50)
    parser.add_argument('--max_chunk_length', type=int, default=150)
    parser.add_argument('--id_column', type=str, default="id")
    parser.add_argument('--cap_zero', action="store_true", help="If set, caps negative similarity values to zero.")
    return parser 


def preprocess(df,args):
    with open("/kaggle/input/final-data/de_stopwords.txt", 'r', encoding='utf-8') as file:
        stopwords_set = {line.strip() for line in file if line.strip()}
        
    def remove_special_characters(text):
        pattern = r'[^a-zA-Z0-9äöüÄÖÜß\s]'
        clean_text = re.sub(pattern, '', text)
        return clean_text

    def remove_stopwords(text, stopwords_set):
        mostly_numeric_pattern = r"^\d*[a-zA-Z]?\d*$"
        return ' '.join([word for word in text.split() if word not in stopwords_set and len(word) > 2 and not re.match(mostly_numeric_pattern, word)])

    df['text'] = df['text'].astype(str)
    df['text'].replace(to_replace=r"\.\.+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\-\-+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"__+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\*\*+", value=" ", regex=True, inplace=True)
    df['text'].replace(to_replace=r"\s+", value=" ", regex=True, inplace=True)
    df['text'] = df['text'].progress_apply(remove_special_characters)
    df['text'] = df['text'].progress_apply(lambda x: remove_stopwords(x, stopwords_set))
    df['length'] = df['text'].progress_apply(lambda x: len(x.split()))
    print(f"Length before dropping all speeches with less than {args.length_threshold} tokens: {len(df)}")
    df = df[df['length'] > args.length_threshold]
    print(f"Length after dropping all speeches with less than {args.length_threshold} tokens: {len(df)}")
    print(f"Average Speech length: {[df['length'].mean()]}")

    # Count top 100 words
    #df['tokens_top100'] = df.text.progress_apply(count_top100)
    #df['fraction_top100'] = df.tokens_top100 / df.length

    # Filter based on top 100 words fraction
    #try:
    #    print('Sample of texts with low fraction of top 100 words:', df[df.fraction_top100 < 0.05].sample(10).text.tolist())
    #    df = df[df.fraction_top100 >= 0.05]
    #    print(f"Number of rows after top 100 words filtering: {len(df)}")
    #except:
    #    print('No rows to sample for top 100 words filtering')
        
    if args.chunk_text:
        def chunk_by_length(x):
            words = x.split()
            if len(words) > args.max_chunk_length:
                chunks = [words[i:i+args.max_chunk_length] for i in range(0, len(words), args.max_chunk_length)]
                last_chunk_length = len(chunks[-1])
                if len(chunks) > 1 and last_chunk_length < args.min_chunk_length:
                    chunks[-2] = chunks[-2] + chunks[-1]
                    del chunks[-1]
                chunked = [" ".join(chunk) for chunk in chunks]
            else:
                chunked = [" ".join(words)]
            return chunked 

        df['text'] = df.text.progress_apply(chunk_by_length)
        df = df.explode("text", ignore_index=True)
        df = df.drop_duplicates(subset=['text']+[f'{args.id_column}'])
        df['chunk_length'] = df.text.progress_apply(lambda x: len(x.split()))
    return df
    


def get_embeddings(text, model):
    #encode text in batches 
    corpus_embeddings = model.encode(text, batch_size=1024, show_progress_bar=True, convert_to_tensor=True)
    assert len(corpus_embeddings) == len(text)
    return corpus_embeddings
    

def length_adjustment_bin(df, length_column, minimum_length=0):
    bins = range(minimum_length, df[length_column].max()+10, 10)
    df[f'{length_column}_bin'] = pd.cut(df[length_column], bins=bins)
    df['evidence_mean'] = df.groupby(f'{length_column}_bin')['evidence_score'].transform('mean')
    df['evidence_adj'] = df['evidence_score'] - df['evidence_mean']
    df['intuition_mean'] = df.groupby(f'{length_column}_bin')['intuition_score'].transform('mean')
    df['intuition_adj'] = df['intuition_score'] - df['intuition_mean']
    return df

def evidence_minus_intuition_score(df):
    df[['evidence_z', 'intuition_z']] = df[['evidence_adj', 'intuition_adj']].apply(zscore)
    df['emi'] = df['evidence_z'] - df['intuition_z']
    return df


# Define a function to calculate similarity and cap negative values beforehand
def calculate_similarity_with_clipping(text_embeddings, lexicon_embedding, cap_zero):
    # Calculate cosine similarity at the word level
    word_similarities = util.cos_sim(text_embeddings, lexicon_embedding)
    
    # Cap negative values at 0 if cap_zero is True
    if cap_zero:
        word_similarities = torch.clamp(word_similarities, min=0)
    
    # Average the capped similarities for each text (row-wise mean)
    avg_similarity_per_text = word_similarities.mean(dim=1)
    return avg_similarity_per_text
    

# Main function
def main(args):
    tqdm.pandas()
    delimiter = '\t' if args.tab_delimiter else None
    
    # Load data
    if args.smoke_test:
        df = pd.read_csv(args.input_file, nrows=500_000, compression=args.compression_type, delimiter=delimiter, dtype={'speech_id': object})
    else:
        df = pd.read_csv(args.input_file, compression=args.compression_type, delimiter=delimiter, dtype={'speech_id': object})
    
    # Rename text column if needed
    if args.text_column != 'text':
        df.rename(columns={args.text_column: 'text'}, inplace=True)
    df['text'] = df['text'].astype(str)
    df = df.drop_duplicates(subset=['text', args.id_column])
    
    # Define stopwords and preprocess
    df = preprocess(df, args)
    print('After pre-processing:', len(df[args.id_column].unique()))

    # Load model
    custom_model = SentenceTransformer(args.model_name_or_path)
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    evidence_sim = torch.Tensor().to(device)
    intuition_sim = torch.Tensor().to(device)
    chunk_size = 500_000
    list_df = [df[idx:idx + chunk_size] for idx in range(0, len(df), chunk_size)]
    
    # For each batch
    for batch in tqdm(list_df):
        batch_text = batch['text'].tolist()
        text_embeddings = get_embeddings(batch_text, custom_model)
        
        # Load and process lexicon embeddings
        evidence_keywords = pd.read_csv(args.evidence_lexicon)['evidence'].tolist()
        evidence_embeddings = get_embeddings(evidence_keywords, custom_model).mean(dim=0).to(device)
        intuition_keywords = pd.read_csv(args.intuition_lexicon)['intuition'][:38].tolist()
        intuition_embeddings = get_embeddings(intuition_keywords, custom_model).mean(dim=0).to(device)

        # Calculate similarity and apply capping if cap_zero is True
        evidence_sim_batch = calculate_similarity_with_clipping(text_embeddings, evidence_embeddings, args.cap_zero)
        intuition_sim_batch = calculate_similarity_with_clipping(text_embeddings, intuition_embeddings, args.cap_zero)

        # Append similarity values to cumulative tensors
        evidence_sim = torch.cat((evidence_sim, evidence_sim_batch), 0)
        intuition_sim = torch.cat((intuition_sim, intuition_sim_batch), 0)

    # Convert similarity tensors to DataFrame columns
    df['evidence_score'] = evidence_sim.cpu().numpy()
    df['intuition_score'] = intuition_sim.cpu().numpy()

    # Adjust scores based on length and compute EMI
    length_column = 'chunk_length'
    df = length_adjustment_bin(df, length_column=length_column, minimum_length=0)
    df = evidence_minus_intuition_score(df)
    
    # Save results
    df.to_csv(args.output_file, index=False, compression=args.compression_type)
    print(df[['evidence_score', 'intuition_score', 'emi']].head())

In [3]:
# Set up arguments manually
class Args:
    model_name_or_path = '/kaggle/input/sbert-model-new/model'
    input_file = '/kaggle/input/german-parliament/speeches_all.csv'
    output_file = 'speeches_all_emi.csv'
    evidence_lexicon = '/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv'
    intuition_lexicon = '/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv'
    save_embeddings = False
    smoke_test = False
    text_column = 'sentence'
    compression_type = 'infer'
    length_threshold = 10
    tab_delimiter = False
    chunk_text = True
    min_chunk_length = 50
    max_chunk_length = 150
    id_column = "id"
    cap_zero = False  # Set this to True to cap negative similarity values at zero


args = Args()

# Run the main function
main(args)

/tmp/ipykernel_23/727962141.py:132: DtypeWarning: Columns (7,8,11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(args.input_file, compression=args.compression_type, delimiter=delimiter, dtype={'speech_id': object})
/tmp/ipykernel_23/727962141.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['text'].replace(to_replace=r"\.\.+", value=" ", regex=True, inplace=True)
/tmp/ipykernel_23/727962141.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or 

Length before dropping all speeches with less than 10 tokens: 8672377
Length after dropping all speeches with less than 10 tokens: 2700156
Average Speech length: [39.69238999524472]


100%|██████████| 2700156/2700156 [00:19<00:00, 140602.41it/s]
/tmp/ipykernel_23/727962141.py:79: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df.text.progress_apply(chunk_by_length)
100%|██████████| 2972331/2972331 [00:12<00:00, 243609.76it/s]


After pre-processing: 329390


/opt/conda/lib/python3.10/site-packages/sentence_transformers/models/WordEmbeddings.py:80: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights = torch.load(os.path.join(in

...

Batches:   0%|          | 0/462 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 6/6 [02:53<00:00, 28.98s/it]
/tmp/ipykernel_23/727962141.py:97: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['evidence_mean'] = df.groupby(f'{length_column}_bin')['evidence_score'].transform('mean')
/tmp/ipykernel_23/727962141.py:99: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df['intuition_mean'] = df.groupby(f'{length_column}_bin')['intuition_score'].transform('mean')


   evidence_score  intuition_score       emi
0       -0.043863        -0.052303  0.180759
1       -0.113884        -0.167698  0.438061
2       -0.117310        -0.128328  0.064490
3       -0.060217        -0.176013  1.070868
4       -0.106117        -0.220753  0.974438
